# 02 — Tek geçişli kaynak bütünlüğü ve test denetimi (inference yok)

**Amaç:** 27 Ekim 2026 tarihli Stage 3 v2 kaynak manifestindeki dosyaları denetlemek, mevcut testleri çalıştırmak ve araştırmanın salt-okunur hazır olma kapısını raporlamak.

Bu notebook **model yüklemez, model yanıtı üretmez, eğitim yapmaz, vaka kabul etmez ve deney iznini açmaz**. GPU gerekmez. Paket kurmaz veya ortamı değiştirmez.

## Kaggle hazırlığı
1. Kaggle Input bölümüne mevcut `dijitalikiz-stage3-source` veri kümesini ekle. İçinde `artifacts/KAGGLE_VALIDATION_SOURCE_BUNDLE_2026-10-27_v2.manifest.json` ve manifestte listelenen kaynaklar bulunmalıdır.
2. Notebook ortamında internet veya GPU açılması gerekmez.
3. Hücreleri sırayla bir kez çalıştır. Çıktı olarak bir JSON denetim raporu ve ZIP arşivi `/kaggle/working` altına yazılır.

## Karar mantığı
- Kaynak dosyaları hash kontrolünden geçmezse testler çalıştırılmaz.
- Manifestteki tek dokümantasyon dosyası `README.md`, 29 Ekim sonrası güncellenmişse bu fark uyarı olarak kaydedilir. Deney kodu, testler, manifestler, şemalar ve analiz dosyalarındaki hash farkları engeldir.
- Test başarısı yalnızca kod/test sözleşmesinin geçtiğini gösterir; vaka geçerliliği veya bilimsel sonuç kanıtı değildir.
- `ready=false` preflight sonucu beklenen bir araştırma blokajı olabilir. Bu, notebook hatası değildir ve inference başlatılmaz.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import importlib.metadata
import json
import os
import platform
import subprocess
import sys
import zipfile

RUN_TS = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
WORK_ROOT = Path('/kaggle/working')
OUT_DIR = WORK_ROOT / f'stage3_no_inference_audit_{RUN_TS}'
OUT_DIR.mkdir(parents=True, exist_ok=False)
REPORT_PATH = OUT_DIR / 'audit_report.json'
TEST_LOG_PATH = OUT_DIR / 'pytest.log'
PREFLIGHT_LOG_PATH = OUT_DIR / 'preflight.log'

def package_version(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None

def sha256_bytes(raw):
    return hashlib.sha256(raw).hexdigest()

def git_blob_sha1(raw):
    header = b'blob ' + str(len(raw)).encode('ascii') + b'\0'
    return hashlib.sha1(header + raw).hexdigest()

def save_report(report):
    REPORT_PATH.write_text(json.dumps(report, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')

runtime = {
    'timestamp_utc': datetime.now(timezone.utc).isoformat(),
    'python': platform.python_version(),
    'platform': platform.platform(),
    'pytest': package_version('pytest'),
    'jsonschema': package_version('jsonschema'),
    'torch': package_version('torch'),
    'transformers': package_version('transformers'),
    'accelerate': package_version('accelerate'),
}

report = {
    'audit_type': 'stage3_source_integrity_and_tests_no_inference',
    'source_bundle_manifest': None,
    'source_commit': None,
    'repo_root': None,
    'runtime': runtime,
    'source_integrity': {'status': 'not_started', 'expected_files': 0, 'matched_files': 0, 'warnings': [], 'blockers': [], 'details': []},
    'tests': {'status': 'not_started', 'pytest_available': runtime['pytest'] is not None, 'test_files': [], 'exit_code': None},
    'research_gate': {'status': 'not_started', 'ready': False, 'accepted_case_count': None, 'accepted_project_count': None, 'blockers': [], 'model_inference_performed': False},
    'model_loaded': False,
    'model_inference_performed': False,
    'model_training_performed': False,
    'case_registry_modified': False,
    'experiment_manifest_modified': False,
    'overall_status': 'BLOCKED',
}

print('Runtime:', json.dumps(runtime, ensure_ascii=False, indent=2))
print('Output directory:', OUT_DIR)


In [ ]:
# v2 manifestini tekil olarak bul. Bulunamazsa başka bir kopyayı tahmin ederek seçme.input_root = Path('/kaggle/input')manifest_rel = Path('artifacts/KAGGLE_VALIDATION_SOURCE_BUNDLE_2026-10-27_v2.manifest.json')manifest_hits = list(input_root.rglob(manifest_rel.name)) if input_root.exists() else []if len(manifest_hits) != 1:    report['source_integrity']['status'] = 'blocked_manifest_not_unique_or_missing'    report['source_integrity']['blockers'].append(        f'Exactly one v2 source-bundle manifest was expected under /kaggle/input; found {len(manifest_hits)}. '        'Attach the current dijitalikiz-stage3-source input containing the v2 bundle.'    )    save_report(report)    raise RuntimeError(report['source_integrity']['blockers'][-1])manifest_path = manifest_hits[0]repo_root = manifest_path.parent.parentreport['source_bundle_manifest'] = str(manifest_path)report['source_commit'] = Nonereport['repo_root'] = str(repo_root)manifest_raw = manifest_path.read_bytes()manifest = json.loads(manifest_raw.decode('utf-8'))report['source_integrity']['manifest_sha256'] = sha256_bytes(manifest_raw)report['source_integrity']['manifest_git_blob_sha1'] = git_blob_sha1(manifest_raw)report['source_commit'] = manifest.get('source_commit')EXPECTED_SOURCE_COMMIT = '99663c2874424feb9cec1d965432030f5cdb3ef8'if report['source_commit'] != EXPECTED_SOURCE_COMMIT:    report['source_integrity']['blockers'].append('v2 manifest source_commit differs from the reviewed 27 October source snapshot.')entries = manifest.get('files')if not isinstance(entries, list) or not entries:    report['source_integrity']['status'] = 'blocked_invalid_manifest'    report['source_integrity']['blockers'].append('Manifest files alanı boş veya geçersiz.')    save_report(report)    raise RuntimeError(report['source_integrity']['blockers'][-1])report['source_integrity']['expected_files'] = len(entries)entry_paths = [entry.get('path') for entry in entries if isinstance(entry, dict)]if len(entry_paths) != len(entries) or len(entry_paths) != len(set(entry_paths)):    report['source_integrity']['blockers'].append('Manifest file paths are malformed or duplicated.')if len(entries) != 31:    report['source_integrity']['blockers'].append(f'Expected 31 v2 manifest entries; found {len(entries)}.')documentation_drift_allowlist = {'README.md'}matched = 0for entry in entries:    rel = entry.get('path') if isinstance(entry, dict) else None    expected_blob = entry.get('git_blob_sha1') if isinstance(entry, dict) else None    expected_bytes = entry.get('byte_length') if isinstance(entry, dict) else None    detail = {'path': rel, 'status': 'blocked'}    if not isinstance(rel, str) or not rel or Path(rel).is_absolute() or '..' in Path(rel).parts:        detail['error'] = 'Unsafe or invalid repository-relative path.'        report['source_integrity']['blockers'].append(f'Invalid manifest path: {rel!r}')        report['source_integrity']['details'].append(detail)        continue    target = repo_root / rel    if not target.is_file():        detail['error'] = 'File missing from Kaggle input.'        report['source_integrity']['blockers'].append(f'Missing source file: {rel}')        report['source_integrity']['details'].append(detail)        continue    raw = target.read_bytes()    actual_blob = git_blob_sha1(raw)    detail.update({'actual_git_blob_sha1': actual_blob, 'expected_git_blob_sha1': expected_blob, 'actual_byte_length': len(raw), 'expected_byte_length': expected_bytes})    if actual_blob == expected_blob and len(raw) == expected_bytes:        detail['status'] = 'match'        matched += 1    elif rel in documentation_drift_allowlist:        detail['status'] = 'documentation_only_drift'        report['source_integrity']['warnings'].append(f'{rel} differs from the v2 snapshot; documentation-only drift is allowed and recorded.')    else:        detail['status'] = 'hash_mismatch'        detail['error'] = 'Source bytes do not match the frozen v2 Git blob SHA-1 and byte length.'        report['source_integrity']['blockers'].append(f'Frozen source mismatch: {rel}')    report['source_integrity']['details'].append(detail)report['source_integrity']['matched_files'] = matchednon_doc_blockers = [x for x in report['source_integrity']['blockers'] if not x.startswith('Expected 31 v2 manifest entries;')]if len(entries) == 31 and not non_doc_blockers:    report['source_integrity']['status'] = 'passed_with_documentation_drift' if report['source_integrity']['warnings'] else 'passed'else:    report['source_integrity']['status'] = 'failed'save_report(report)print('Source integrity:', report['source_integrity']['status'])print('Manifest source commit:', report['source_commit'])print('Matched files:', matched, '/', len(entries))print('Warnings:', report['source_integrity']['warnings'])print('Blockers:', report['source_integrity']['blockers'])if report['source_integrity']['status'] == 'failed':    print('Hash kontrolü başarısız; testler güvenlik nedeniyle çalıştırılmayacak.')

In [ ]:
# Manifestteki gerçek test dosyalarını çalıştır. Paket kurulumu veya yükseltme yapılmaz.if report['source_integrity']['status'] == 'failed':    report['tests']['status'] = 'not_run_source_integrity_failed'    save_report(report)else:    test_paths = sorted({entry['path'] for entry in entries if isinstance(entry, dict) and isinstance(entry.get('path'), str) and entry['path'].startswith('tests/') and entry['path'].endswith('.py')})    report['tests']['test_files'] = test_paths    if not test_paths:        report['tests']['status'] = 'blocked_no_test_paths_in_manifest'        report['tests']['error'] = 'Manifest contains no test Python files.'    elif len(test_paths) != 9:        report['tests']['status'] = 'blocked_unexpected_test_file_count'        report['tests']['error'] = f'Expected 9 test modules in the reviewed v2 manifest; found {len(test_paths)}.'    elif runtime['pytest'] is None:        report['tests']['status'] = 'blocked_pytest_not_installed'        report['tests']['error'] = 'pytest is not installed; notebook intentionally does not install or upgrade packages.'    elif runtime['jsonschema'] is None:        report['tests']['status'] = 'blocked_jsonschema_not_installed'        report['tests']['error'] = 'jsonschema is not installed; notebook intentionally does not install or upgrade packages.'    else:        command = [sys.executable, '-m', 'pytest', '-q', *test_paths]        report['tests']['command'] = command        report['tests']['working_directory'] = str(repo_root)        try:            proc = subprocess.run(command, cwd=str(repo_root), capture_output=True, text=True, errors='replace', timeout=900, check=False)            test_output = (proc.stdout or '') + ('\n--- STDERR ---\n' + proc.stderr if proc.stderr else '')            TEST_LOG_PATH.write_text(test_output, encoding='utf-8')            report['tests']['exit_code'] = proc.returncode            report['tests']['status'] = 'passed' if proc.returncode == 0 else 'failed'            report['tests']['log_path'] = str(TEST_LOG_PATH)        except subprocess.TimeoutExpired as exc:            TEST_LOG_PATH.write_text(str(exc.stdout or '') + '\nTIMEOUT after 900 seconds\n', encoding='utf-8')            report['tests']['status'] = 'timed_out'            report['tests']['log_path'] = str(TEST_LOG_PATH)        except Exception as exc:            TEST_LOG_PATH.write_text(f'{type(exc).__name__}: {exc}\n', encoding='utf-8')            report['tests']['status'] = 'execution_error'            report['tests']['error'] = f'{type(exc).__name__}: {exc}'            report['tests']['log_path'] = str(TEST_LOG_PATH)    save_report(report)    print('Test status:', report['tests']['status'])    print('Test files:', len(report['tests']['test_files']))    print('Test exit code:', report['tests']['exit_code'])    if TEST_LOG_PATH.exists():        print(TEST_LOG_PATH.read_text(encoding='utf-8')[-12000:])

In [ ]:
# Kanonik preflight salt okunurdur: vakaları/modeli çalıştırmaz, yalnızca araştırma engellerini listeler.
if report['source_integrity']['status'] == 'failed':
    report['research_gate']['status'] = 'not_run_source_integrity_failed'
else:
    preflight_script = repo_root / 'scripts' / 'evidence_abstention_preflight.py'
    if not preflight_script.is_file():
        report['research_gate']['status'] = 'blocked_preflight_script_missing'
        report['research_gate']['blockers'] = ['scripts/evidence_abstention_preflight.py is missing.']
    else:
        command = [sys.executable, str(preflight_script), '--root', str(repo_root)]
        try:
            proc = subprocess.run(command, cwd=str(repo_root), capture_output=True, text=True, errors='replace', timeout=120, check=False)
            gate_output = (proc.stdout or '') + ('\n--- STDERR ---\n' + proc.stderr if proc.stderr else '')
            PREFLIGHT_LOG_PATH.write_text(gate_output, encoding='utf-8')
            try:
                gate = json.loads(proc.stdout)
            except json.JSONDecodeError:
                gate = None
            if not isinstance(gate, dict):
                report['research_gate']['status'] = 'invalid_preflight_output'
                report['research_gate']['blockers'] = ['Preflight did not return a JSON object. Inspect preflight.log.']
            else:
                report['research_gate'].update({
                    'status': 'ready' if gate.get('ready') is True else 'blocked_as_expected',
                    'ready': gate.get('ready') is True,
                    'accepted_case_count': gate.get('accepted_case_count'),
                    'accepted_project_count': gate.get('accepted_project_count'),
                    'blocker_count': gate.get('blocker_count'),
                    'blockers': gate.get('blockers', []),
                    'model_inference_performed': gate.get('model_inference_performed', False),
                    'preflight_exit_code': proc.returncode,
                    'log_path': str(PREFLIGHT_LOG_PATH),
                })
        except subprocess.TimeoutExpired:
            report['research_gate']['status'] = 'timed_out'
            report['research_gate']['blockers'] = ['Read-only preflight timed out.']
        except Exception as exc:
            report['research_gate']['status'] = 'execution_error'
            report['research_gate']['blockers'] = [f'{type(exc).__name__}: {exc}']

save_report(report)
print('Research gate status:', report['research_gate']['status'])
print('Research ready:', report['research_gate']['ready'])
print('Accepted cases:', report['research_gate']['accepted_case_count'])
print('Accepted projects:', report['research_gate']['accepted_project_count'])
print('Research blockers:', len(report['research_gate']['blockers']))
for item in report['research_gate']['blockers']:
    print('-', item)


In [ ]:
# Nihai rapor ve tek ZIP çıktısı.
test_ok = report['tests']['status'] == 'passed'
source_ok = report['source_integrity']['status'] in {'passed', 'passed_with_documentation_drift'}
if source_ok and test_ok:
    report['overall_status'] = 'CODE_VALIDATION_PASSED_RESEARCH_GATE_' + ('READY' if report['research_gate']['ready'] else 'BLOCKED')
elif not source_ok:
    report['overall_status'] = 'SOURCE_INTEGRITY_FAILED'
elif report['tests']['status'].startswith('blocked'):
    report['overall_status'] = 'SOURCE_INTEGRITY_PASSED_TESTS_BLOCKED'
else:
    report['overall_status'] = 'SOURCE_INTEGRITY_PASSED_TESTS_FAILED'
report['model_loaded'] = False
report['model_inference_performed'] = False
report['model_training_performed'] = False
report['case_registry_modified'] = False
report['experiment_manifest_modified'] = False
save_report(report)
zip_path = WORK_ROOT / f'stage3_no_inference_audit_{RUN_TS}.zip'
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(OUT_DIR.rglob('*')):
        if path.is_file():
            archive.write(path, arcname=str(path.relative_to(OUT_DIR)))
print('\nFINAL STATUS:', report['overall_status'])
print('Report:', REPORT_PATH)
print('Bundle:', zip_path)
print('IMPORTANT: This audit does not accept cases or authorize any model inference.')
